# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [5]:
# model_name = "canopylabs/3b-hi-ft-research_release" # -> probably this is the better one to finetune on
# model_name = "/cache/orpheus-merged-vllm"
# model_name = "rumik-ai/orpheus-ira-hinglish-full"
# model_name = "rumik-ai/orpheus-ira-hinglish-lora2"
# model_name = "rumik-ai/orpheus-ira-hinglish-lora2"
model_name = "rumik-ai/ira-160h-64lr5e-5"

# model_name = "canopylabs/3b-hi-pretrain-research_release"

In [6]:
import os
os.listdir('/mnt/orpheus-cache')

['canopy-hindi-3b',
 'ira-160h-64lr5e-5',
 'orpheus-3b',
 'orpheus-finetuned-pretrain',
 'orpheus-finetuned-text',
 'orpheus-instruction-finetuned',
 'orpheus-ira-hinglish',
 'orpheus-ira-hinglish-lora2',
 'orpheus-merged-vllm',
 'output_22d00f4b.wav',
 'output_277ec293.wav',
 'output_43e7511b.wav',
 'output_f6f667c0.wav']

In [4]:
# model_path = "/mnt/orpheus-cache/orpheus-merged-vllm"

In [3]:
%uv pip install torch numpy soundfile librosa

Using Python 3.12.6 environment at: /usr/local
Resolved 49 packages in 205ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
pooch      ------------------------------ 14.92 KiB/63.06 KiB
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
pooch      ------------------------------ 14.92 KiB/63.06 KiB
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
pooch      ------------------------------ 14.92 KiB/63.06 KiB
librosa    ------------------------------ 16.00 KiB/254.64 KiB
⠙ Preparing packages... (0/5)
audioread  ------------------------------ 14.88 KiB/22.60 KiB
pooch      ------------------------------ 14.92 KiB/63.06 KiB


In [8]:
import os 
os.listdir('/mnt/orpheus-cache/orpheus-merged-vllm')

['chat_template.jinja',
 'config.json',
 'generation_config.json',
 'model-00001-of-00002.safetensors',
 'model-00002-of-00002.safetensors',
 'model.safetensors.index.json',
 'special_tokens_map.json',
 'tokenizer.json',
 'tokenizer_config.json']

In [9]:
import json
with open("/mnt/orpheus-cache/orpheus-merged-vllm/config.json", "r") as f:
    config = json.load(f)

with open("/mnt/orpheus-cache/orpheus-merged-vllm/generation_config.json", "r") as f:
    generation_config = json.load(f)

with open("/mnt/orpheus-cache/orpheus-merged-vllm/tokenizer_config.json", "r") as f:
    tok_config = json.load(f)

with open("/mnt/orpheus-cache/orpheus-merged-vllm/tokenizer.json", "r") as f:
    tok = json.load(f)

In [10]:
config

{'architectures': ['LlamaForCausalLM'],
 'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 128000,
 'dtype': 'float16',
 'eos_token_id': 128001,
 'head_dim': 128,
 'hidden_act': 'silu',
 'hidden_size': 3072,
 'initializer_range': 0.02,
 'intermediate_size': 8192,
 'max_position_embeddings': 131072,
 'mlp_bias': False,
 'model_type': 'llama',
 'num_attention_heads': 24,
 'num_hidden_layers': 28,
 'num_key_value_heads': 8,
 'pretraining_tp': 1,
 'rms_norm_eps': 1e-05,
 'rope_scaling': {'factor': 32.0,
  'high_freq_factor': 4.0,
  'low_freq_factor': 1.0,
  'original_max_position_embeddings': 8192,
  'rope_type': 'llama3'},
 'rope_theta': 500000.0,
 'tie_word_embeddings': True,
 'transformers_version': '4.57.3',
 'use_cache': True,
 'vocab_size': 156971}

In [58]:
#@title Installation & Setup
%uv pip install snac ipywebrtc
from snac import SNAC
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, AutoTokenizer
import numpy as np
import soundfile as sf
import IPython.display as ipd
import librosa
from ipywebrtc import AudioRecorder, Audio
from IPython.display import display
import ipywidgets as widgets
import random
import os

def set_seed(seed_value=42):
    """Set seed for reproducibility."""
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    
    # If using CUDA (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # For multi-GPU
        # Optional: set for deterministic algorithms (might impact performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(4)

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz")
snac_model = snac_model.to("cpu")

print("We have loaded the tokeniser/detokeniser model to the cpu, to use vram - use the gpu for faster inference")

tokeniser_name = "meta-llama/Llama-3.2-3B-Instruct"
from huggingface_hub import snapshot_download

# Download only model config and safetensors
model_path = snapshot_download(
    repo_id=model_name,
    token=True,
    allow_patterns=[
        "config.json",
        "*.safetensors",
        "model.safetensors.index.json",
    ],
    ignore_patterns=[
        "optimizer.pt",
        "pytorch_model.bin",
        "training_args.bin",
        "scheduler.pt",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.json",
        "merges.txt",
        "tokenizer.*"
    ]
)

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
model.cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)

# engine_args = AsyncEngineArgs(
#     model=model_name,
#     tokenizer=model_name,
#     # tokenizer_kwargs={
#     #     "pad_token_id": 128263,
#     # },
#     trust_remote_code=True,
#     dtype="float16",
#     tensor_parallel_size=1,
#     gpu_memory_utilization=0.85,
#     enable_prefix_caching=True,
# )

# engine = AsyncLLMEngine.from_engine_args(engine_args)
# tokenizer = engine.tokenizer

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 8ms


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.
We have loaded the tokeniser/detokeniser model to the cpu, to use vram - use the gpu for faster inference


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [59]:
# prompts = [
#     "Hey there my name is Tara, <chuckle> and I'm a speech generation model that can sound like a person.",
#     "I've also been taught to understand and produce paralinguistic things like sighing, or chuckling, or yawning!",
#     "I live in San Francisco, and have, uhm let's see, 3 billion 7 hundred ... well, lets just say a lot of parameters.",
# ]

# prompts = [
#     "नमस्ते, मेरा नाम तारा है। <giggles> मैं एक स्पीच-जनरेशन मॉडल हूँ जो इंसानों की तरह बोल भी सकती हूँ और सुन भी सकती हूँ।",
#     "मुझे ऐसे-वैसे बोलने की नहीं, बल्कि आह भरना, <giggles>, जम्हाई लेना जैसी अभिव्यक्तियाँ भी समझने और बनाने की ट्रेनिंग दी गई है!",
#     "मैं तो सैन फ़्रांसिस्को में रहती हूँ, और मेरे पास… उhm… चलो मान लो बहुत ही ज़्यादा पैरामीटर्स हैं, गिनने बैठो तो समय ही निकल जाए!",
# ]

prompts = [
    # "नमस्ते, मेरा नाम तारा है। <chuckle> मैं एक स्पीच-जनरेशन मॉडल हूँ जो इंसानों की तरह बोल भी सकती हूँ और सुन भी सकती हूँ।",
    # "मुझे ऐसे-वैसे बोलने की नहीं, बल्कि आह भरना, <chuckle>, जम्हाई लेना जैसी अभिव्यक्तियाँ भी समझने और बनाने की ट्रेनिंग दी गई है!",
    # "मैं तो सैन फ़्रांसिस्को में रहती हूँ, और मेरे पास… उhm… चलो मान लो बहुत ही ज़्यादा पैरामीटर्स हैं, गिनने बैठो तो समय ही निकल जाए!",
    "नमस्ते, मेरा नाम तारा है।",
    "नमस्ते, आज का मौसम बहुत अच्छा है। मैं एक हिंदी टेक्स्ट टू स्पीच मॉडल हूं और मैं आपकी मदद करने के लिए यहां हूं।",
    "hi how are you doing",
    "चलो साफ़ कर देते हैं—कोई रहस्य नहीं, बस व्याकरण का खेल है।",
    "ज़िंदगी भी कुछ ऐसी ही है। हर दिन हम छोटे छोटे वाक्य बोलते हैं, छोटे फैसले लेते हैं, और उन्हीं से हमारी कहानी बनती है। कभी हम आत्मविश्वास से भरे होते हैं, कभी संदेह में। कभी हम कहते हैं कि हम कर सकते हैं, और कभी हमें खुद को मनाना पड़ता है कि हम कर सकते हैं। सीख यही है कि भाषा की तरह जीवन में भी स्पष्टता ज़रूरी है। अपने विचारों में, अपने शब्दों में, और अपने इरादों में। जब मन साफ़ होता है, तो आवाज़ भी साफ़ होती है। धन्यवाद।",
    "hi how are you, hello what are you doing",
    "Are you okay? <yawn> What happened?",
    "Hello! <laugh> That's so funny!",
    "<sigh> I don't know what to do.",
    "I'm so tired today. <yawn> Need some coffee.",
    "aap kaise ho ?"
    
]

chosen_voice = "tara" # see github for other voices

In [60]:
for prompt in prompts:
    # print(prompt)
    print(tokenizer(prompt, return_tensors="pt"))

{'input_ids': tensor([[128000,  61196,  88344,  79468, 100365, 103608,  92317, 101517,  24810,
         100282, 100497, 100329, 100273,  24810,  85410, 100436]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
{'input_ids': tensor([[128000,  61196,  88344,  79468, 100365, 103608, 105116,  48909,  24810,
          92317, 100712, 113731, 102875, 102223, 105966, 104585,  24810,  85410,
         100436,  92317, 100400, 100549,  85410, 100929, 100296,  44747, 101002,
         100907, 100782, 100431, 101002, 100295,  69258, 101242, 104138,  92317,
         119376,  92911,  85410, 105684, 100358,  92317, 100400, 102168,  44747,
         111832, 100855,  35470,  48909,  35470, 100293, 100471, 100866, 100515,
          85410, 105684, 100278]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])}
{'in

In [69]:
#@title Format prompts into correct template

prompts = [f"ira: {p}" for p in prompts]

all_input_ids = []

for prompt in prompts:
  input_ids = tokenizer(prompt, return_tensors="pt").input_ids
  all_input_ids.append(input_ids)

start_token = torch.tensor([[ 128259]], dtype=torch.int64) # Start of human
end_tokens = torch.tensor([[128009, 128260]], dtype=torch.int64) # End of text, End of human

all_modified_input_ids = []
for input_ids in all_input_ids:
  modified_input_ids = torch.cat([start_token, input_ids, end_tokens], dim=1) # SOH SOT Text EOT EOH
  all_modified_input_ids.append(modified_input_ids)

all_padded_tensors = []
all_attention_masks = []
max_length = max([modified_input_ids.shape[1] for modified_input_ids in all_modified_input_ids])
for modified_input_ids in all_modified_input_ids:
  padding = max_length - modified_input_ids.shape[1]
  padded_tensor = torch.cat([torch.full((1, padding), 128263, dtype=torch.int64), modified_input_ids], dim=1)
  attention_mask = torch.cat([torch.zeros((1, padding), dtype=torch.int64), torch.ones((1, modified_input_ids.shape[1]), dtype=torch.int64)], dim=1)
  all_padded_tensors.append(padded_tensor)
  all_attention_masks.append(attention_mask)

all_padded_tensors = torch.cat(all_padded_tensors, dim=0)
all_attention_masks = torch.cat(all_attention_masks, dim=0)

input_ids = all_padded_tensors.to("cuda")
attention_mask = all_attention_masks.to("cuda")

In [70]:
#@title Generate Output
print("*** Model.generate is slow - see vllm implementation on github for realtime streaming and inference")
print("*** Increase/decrease inference params for more expressive less stable generations")

with torch.no_grad():
  generated_ids = model.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=12000,
      do_sample=True,
      temperature=0.6,
      top_p=0.95,
      repetition_penalty=1.1,
      num_return_sequences=1,
      eos_token_id=128258,
  )

Setting `pad_token_id` to `eos_token_id`:128258 for open-end generation.


*** Model.generate is slow - see vllm implementation on github for realtime streaming and inference
*** Increase/decrease inference params for more expressive less stable generations


In [71]:
#@title Parse Output as speech
token_to_find = 128257
token_to_remove = 128258

token_indices = (generated_ids == token_to_find).nonzero(as_tuple=True)

if len(token_indices[1]) > 0:
    last_occurrence_idx = token_indices[1][-1].item()
    cropped_tensor = generated_ids[:, last_occurrence_idx+1:]
else:
    cropped_tensor = generated_ids

mask = cropped_tensor != token_to_remove

processed_rows = []

for row in cropped_tensor:
    masked_row = row[row != token_to_remove]
    processed_rows.append(masked_row)

code_lists = []

for row in processed_rows:
    row_length = row.size(0)
    new_length = (row_length // 7) * 7
    trimmed_row = row[:new_length]
    trimmed_row = [t - 128266 for t in trimmed_row]
    code_lists.append(trimmed_row)


def redistribute_codes(code_list):
  layer_1 = []
  layer_2 = []
  layer_3 = []
  for i in range((len(code_list)+1)//7):
    layer_1.append(code_list[7*i])
    layer_2.append(code_list[7*i+1]-4096)
    layer_3.append(code_list[7*i+2]-(2*4096))
    layer_3.append(code_list[7*i+3]-(3*4096))
    layer_2.append(code_list[7*i+4]-(4*4096))
    layer_3.append(code_list[7*i+5]-(5*4096))
    layer_3.append(code_list[7*i+6]-(6*4096))
  codes = [torch.tensor(layer_1).unsqueeze(0),
         torch.tensor(layer_2).unsqueeze(0),
         torch.tensor(layer_3).unsqueeze(0)]
  audio_hat = snac_model.decode(codes)
  return audio_hat

my_samples = []
for code_list in code_lists:
  samples = redistribute_codes(code_list)
  my_samples.append(samples)


In [72]:
#### **** debugging **** ####
def redistribute_codes(code_list):
  layer_1 = []
  layer_2 = []
  layer_3 = []
  for i in range((len(code_list)+1)//7):
    layer_1.append(code_list[7*i])
    layer_2.append(code_list[7*i+1]-4096)
    layer_3.append(code_list[7*i+2]-(2*4096))
    layer_3.append(code_list[7*i+3]-(3*4096))
    layer_2.append(code_list[7*i+4]-(4*4096))
    layer_3.append(code_list[7*i+5]-(5*4096))
    layer_3.append(code_list[7*i+6]-(6*4096))

  return layer_1, layer_2, layer_3

In [73]:
# code_lists[0]
l1, l2, l3 = redistribute_codes(code_list)

In [74]:
with torch.no_grad():
    print([t.item() for t in l1])
    print([t.item() for t in l2])
    print([t.item() for t in l3])

[717, 1260, 1902, 3061, 3507, 468, 3195, 1205, 3060, 180, 2339, 19]
[3562, 2259, 1177, 3442, 1869, 2463, 390, 1769, 1939, 2492, 3822, 4037, 3916, 2316, 3362, 2385, 448, 3589, 725, 791, 3967, 888, 1053, 956]
[2825, 1762, 2061, 1152, 2455, 3713, 1143, 1933, 524, 209, 57, 963, 2527, 3224, 3592, 301, 237, 3355, 93, 1384, 3185, 2018, 2580, 3537, 3958, 3958, 605, 3668, 4083, 3912, 2957, 665, 3167, 3853, 2150, 2641, 1737, 2105, 3472, 2344, 3192, 1367, 1634, 217, 1618, 178, 4018, 2825]


In [75]:
my_samples[0].shape

torch.Size([1, 1, 65536])

In [76]:
#@title Display Audio
from IPython.display import display, Audio
if len(prompts) != len(my_samples):
  raise Exception("Number of prompts and samples do not match")
else:
  for i in range(len(my_samples)):
    print(prompts[i])
    samples = my_samples[i]
    display(Audio(samples.detach().squeeze().to("cpu").numpy(), rate=24000))


ira: नमस्ते, मेरा नाम तारा है।


ira: नमस्ते, आज का मौसम बहुत अच्छा है। मैं एक हिंदी टेक्स्ट टू स्पीच मॉडल हूं और मैं आपकी मदद करने के लिए यहां हूं।


ira: hi how are you doing


ira: चलो साफ़ कर देते हैं—कोई रहस्य नहीं, बस व्याकरण का खेल है।


ira: ज़िंदगी भी कुछ ऐसी ही है। हर दिन हम छोटे छोटे वाक्य बोलते हैं, छोटे फैसले लेते हैं, और उन्हीं से हमारी कहानी बनती है। कभी हम आत्मविश्वास से भरे होते हैं, कभी संदेह में। कभी हम कहते हैं कि हम कर सकते हैं, और कभी हमें खुद को मनाना पड़ता है कि हम कर सकते हैं। सीख यही है कि भाषा की तरह जीवन में भी स्पष्टता ज़रूरी है। अपने विचारों में, अपने शब्दों में, और अपने इरादों में। जब मन साफ़ होता है, तो आवाज़ भी साफ़ होती है। धन्यवाद।


...

ira: Are you okay? <yawn> What happened?


ira: Hello! <laugh> That's so funny!


ira: <sigh> I don't know what to do.


ira: I'm so tired today. <yawn> Need some coffee.


ira: aap kaise ho ?


In [55]:
# <laugh>, <chuckle>, <sigh>, <cough>, <sniffle>, <groan>, <yawn>, <gasp>